# Experiment 7 — CTGAN per Segment

Trains a **Conditional Tabular GAN (CTGAN)** on each K-Means segment separately using mode-specific normalization, then samples proportionally.

**Requirements:** GPU runtime strongly recommended for CTGAN. Upload `REAL_DATA.xlsx` when prompted.

## 1. Install

In [ ]:
!pip install --quiet sdv
print('SDV installed.')

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
SEED = 42
np.random.seed(SEED)
print('Ready.')

## 3. Upload real data

In [ ]:
from google.colab import files
uploaded = files.upload()
REAL_FILE = next(iter(uploaded))
real = pd.read_excel(REAL_FILE)
print(f'Real data: {real.shape}')
real.head()

## 4. K-Means segmentation (K=5)

In [ ]:
real_enc = real.copy()
for col in real.columns:
    real_enc[col] = LabelEncoder().fit_transform(real[col].astype(str))

km = KMeans(n_clusters=5, random_state=SEED, n_init=50)
real['segment'] = km.fit_predict(real_enc.values)

shares = {c: len(real[real['segment']==c])/len(real) for c in range(5)}
print('Segment shares:', {k: f'{v:.2f}' for k,v in shares.items()})

## 5. Fit CTGAN per segment + sample

In [ ]:
N_SYNTH = 274

n_per = {}
rem = N_SYNTH
for c in range(4):
    n_per[c] = int(round(shares[c] * N_SYNTH)); rem -= n_per[c]
n_per[4] = rem

feature_cols = [c for c in real.columns if c != 'segment']
pieces = []

for c in range(5):
    sub = real[real['segment'] == c][feature_cols].reset_index(drop=True)
    print(f'Segment {c}: fitting CTGAN on n={len(sub)}...')

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(sub)
    for col in sub.columns:
        metadata.update_column(column_name=col, sdtype='categorical')

    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=500,          # increase to 800 for better quality
        batch_size=50,       # small batch for small cluster sizes
        verbose=False
    )
    synthesizer.fit(sub)
    synth_chunk = synthesizer.sample(num_rows=n_per[c])
    pieces.append(synth_chunk)
    print(f'  Sampled {n_per[c]} rows.')

synth = pd.concat(pieces, ignore_index=True)
synth = synth.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nTotal synthetic respondents: {len(synth)}')
synth.head()

## 6. Save and download

In [ ]:
OUT = 'exp7_ctgan_segments.xlsx'
synth.to_excel(OUT, index=False)
print(f'Saved {OUT}')
from google.colab import files
files.download(OUT)

## 7. Sanity check

In [ ]:
print('Q1. Age Group:')
print(synth['Q1. Age Group'].value_counts())
print('\nQ11. Plan to Shop Again:')
print(synth['Q11. Plan to Shop Again in Future'].value_counts())